# LoRA From Scratch (no peft library)

Companion notebook for the [Parameter-Efficient Fine-Tuning](https://codemia.io/courses/llm_inside_and_out/parameter_efficient_fine_tuning) lesson on Codemia.

**What you will build**: LoRA adapters on GPT-2's attention linears, implemented by hand. You will freeze every base-model parameter, wire the adapters into the forward pass, and verify by inspecting `.grad` that only adapter weights actually train. Then you will compare the resulting task accuracy against a full fine-tune on the same data.

**Time**: ~20-30 minutes on T4

**GPU required**: yes

## Setup

In [ ]:
%pip install -q torch transformers datasets matplotlib

In [ ]:
import math, time, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Concept recap

LoRA (Hu et al., 2021) freezes the base model weight `W` and adds a low-rank delta:

```
output = x @ W + x @ (A @ B) * alpha/r
```

where `A` is `(in, r)` and `B` is `(r, out)`, with rank `r` typically 4-16. The full model sees `W + A B` as the effective weight, but only `A` and `B` are trainable. That's a ~1000x reduction in trainable parameters for a large model.

Crucially, because `W` is frozen, the base model cannot *forget* -- its pretrained knowledge is preserved byte-for-byte. At inference time you can either keep the adapter separate (for swapping in and out multiple tasks) or fold it into `W` for zero runtime overhead.

## Implementation: a LoRA-wrapped Linear

In [ ]:
class LoRALinear(nn.Module):
    # Wraps an existing nn.Linear with a low-rank delta.
    # Base weight and bias are frozen; only A and B are trainable.
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base_linear
        # Freeze the base parameters. We do this here rather than relying on the caller.
        for p in self.base.parameters():
            p.requires_grad = False

        in_features = base_linear.in_features
        out_features = base_linear.out_features

        # LoRA matrices. Convention from the paper:
        #   A is initialized with Kaiming (standard random init)
        #   B is initialized to ZERO so the initial delta is zero.
        #   This guarantees the LoRA-wrapped model starts exactly equal to the frozen base.
        self.lora_A = nn.Parameter(torch.zeros(in_features, r))
        self.lora_B = nn.Parameter(torch.zeros(r, out_features))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        # self.lora_B stays zero. That's the point.

        self.scaling = alpha / r
        self.r = r

    def forward(self, x):
        # Frozen base forward + low-rank delta.
        base_out = self.base(x)
        # (x @ A) @ B: we do it in two steps to keep the inner dim at r, which is cheap.
        delta = (x @ self.lora_A) @ self.lora_B * self.scaling
        return base_out + delta

## Wire LoRA into GPT-2

GPT-2's attention uses a single fused `c_attn` linear that produces `[q, k, v]` concatenated. We wrap it, along with the attention output projection `c_proj`. That's where LoRA traditionally goes in the LoRA paper. We leave the MLP alone for a cleaner demo.

In [ ]:
def inject_lora(model, r=8, alpha=16):
    # Walk every transformer block and replace c_attn / c_proj with LoRALinear.
    # GPT-2's c_attn is a 'Conv1D' in transformers, which behaves like nn.Linear for our purposes
    # but we need to construct a matching nn.Linear to wrap -- or we wrap the Conv1D directly.
    # Simpler: monkey-patch the forward with a delta computation.
    from transformers.pytorch_utils import Conv1D

    class LoRAConv1D(nn.Module):
        def __init__(self, base: Conv1D, r, alpha):
            super().__init__()
            self.base = base
            for p in self.base.parameters():
                p.requires_grad = False
            # Conv1D weight shape is (in, out), same as we'd use for A@B.
            in_features = base.weight.shape[0]
            out_features = base.weight.shape[1]
            self.lora_A = nn.Parameter(torch.zeros(in_features, r))
            self.lora_B = nn.Parameter(torch.zeros(r, out_features))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            self.scaling = alpha / r

        def forward(self, x):
            base_out = self.base(x)
            delta = (x @ self.lora_A) @ self.lora_B * self.scaling
            return base_out + delta

    n_wrapped = 0
    for block in model.transformer.h:
        attn = block.attn
        attn.c_attn = LoRAConv1D(attn.c_attn, r, alpha)
        attn.c_proj = LoRAConv1D(attn.c_proj, r, alpha)
        n_wrapped += 2
    return n_wrapped

## Load GPT-2 and inject

In [ ]:
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

# Freeze everything first, then inject_lora will un-freeze only the A/B parameters it creates.
for p in model.parameters():
    p.requires_grad = False

n_wrapped = inject_lora(model, r=8, alpha=16)
model.to(device)
print(f"wrapped {n_wrapped} linears with LoRA")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total params:     {total:,}")
print(f"trainable params: {trainable:,}  ({trainable/total:.2%})")

## Proof: only adapter weights have gradients

Run a forward + backward with a dummy loss. Any parameter with a non-None grad is one that will be updated by the optimizer.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

ids = tokenizer("The capital of France is", return_tensors='pt').input_ids.to(device)
# Train-mode forward with labels = self to get a real LM loss.
out = model(ids, labels=ids)
out.loss.backward()

with_grad = 0
no_grad = 0
lora_params = []
base_params_nonzero_grad = []
for name, p in model.named_parameters():
    if p.grad is not None and p.grad.abs().sum() > 0:
        with_grad += 1
        if 'lora' in name:
            lora_params.append(name)
        else:
            base_params_nonzero_grad.append(name)
    else:
        no_grad += 1
print(f"params with gradient: {with_grad}")
print(f"params without:       {no_grad}")
print(f"sample LoRA params with grad (first 4): {lora_params[:4]}")
print(f"base params with grad: {len(base_params_nonzero_grad)}  (should be 0)")

## Fine-tune on SST-2

In [ ]:
sst2 = load_dataset('glue', 'sst2')

pos_id = tokenizer.encode(' positive')[0]
neg_id = tokenizer.encode(' negative')[0]

def format_example(text, label=None):
    prompt = f"Review: {text.strip()}\nSentiment:"
    if label is None:
        return prompt
    target = ' positive' if label == 1 else ' negative'
    return prompt + target

def prepare_batch(examples, max_len=96):
    prompts = [format_example(e['sentence'], None) for e in examples]
    fulls = [format_example(e['sentence'], e['label']) for e in examples]
    enc_full = tokenizer(fulls, return_tensors='pt', padding=True, truncation=True, max_length=max_len)
    enc_prompt = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=max_len)
    input_ids = enc_full.input_ids.to(device)
    attention_mask = enc_full.attention_mask.to(device)
    labels = input_ids.clone()
    for i in range(len(examples)):
        plen = int(enc_prompt.attention_mask[i].sum().item())
        labels[i, :plen] = -100
        labels[i, attention_mask[i] == 0] = -100
    return input_ids, attention_mask, labels

In [ ]:
@torch.no_grad()
def eval_accuracy(m, n=200):
    m.eval()
    correct = 0
    for i in range(n):
        ex = sst2['validation'][i]
        prompt = format_example(ex['sentence'], None)
        ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
        logits = m(ids).logits[0, -1]
        pred = 1 if logits[pos_id] > logits[neg_id] else 0
        correct += int(pred == ex['label'])
    m.train()
    return correct / n

In [ ]:
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,  # LoRA can use a much higher LR than full fine-tuning
)

print(f"LoRA baseline acc: {eval_accuracy(model):.3f}")

import random
random.seed(0)
train_idx = list(range(len(sst2['train'])))
random.shuffle(train_idx)

lora_history = []
for step in range(1, 201):
    batch = [sst2['train'][train_idx[(step * 8 + j) % len(train_idx)]] for j in range(8)]
    input_ids, attention_mask, labels = prepare_batch(batch)
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    opt.zero_grad(set_to_none=True)
    out.loss.backward()
    opt.step()
    if step % 40 == 0:
        acc = eval_accuracy(model)
        lora_history.append((step, acc))
        print(f"step {step:4d}  loss {out.loss.item():.3f}  acc {acc:.3f}")

## Baseline: full fine-tune at a reasonable LR

In [ ]:
# Fresh model, all params unfrozen. Use the lower LR that keeps forgetting manageable.
full_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
full_opt = torch.optim.AdamW(full_model.parameters(), lr=1e-5)

random.seed(0)
train_idx = list(range(len(sst2['train'])))
random.shuffle(train_idx)

print(f"full baseline acc: {eval_accuracy(full_model):.3f}")
full_history = []
for step in range(1, 201):
    batch = [sst2['train'][train_idx[(step * 8 + j) % len(train_idx)]] for j in range(8)]
    input_ids, attention_mask, labels = prepare_batch(batch)
    out = full_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    full_opt.zero_grad(set_to_none=True)
    out.loss.backward()
    full_opt.step()
    if step % 40 == 0:
        acc = eval_accuracy(full_model)
        full_history.append((step, acc))
        print(f"step {step:4d}  loss {out.loss.item():.3f}  acc {acc:.3f}")

## Compare learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
if lora_history:
    xs, ys = zip(*lora_history); ax.plot(xs, ys, 'o-', label='LoRA (r=8)')
if full_history:
    xs, ys = zip(*full_history); ax.plot(xs, ys, 's-', label='full fine-tune')
ax.set_xlabel('step')
ax.set_ylabel('SST-2 accuracy')
ax.set_title('LoRA vs full fine-tune on the same task')
ax.legend()
plt.tight_layout()
plt.show()

n_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_full = sum(p.numel() for p in full_model.parameters())
print(f"LoRA trainable params: {n_lora:,}  ({n_lora/n_full:.2%} of full)")
print(f"full trainable params: {n_full:,}")

## Variation 1: rank sweep

Different ranks give different capacity-efficiency tradeoffs. Sweep r in {2, 4, 8, 16} and observe when the returns diminish.

In [ ]:
def train_lora_at_rank(r, n_steps=150):
    m = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
    for p in m.parameters():
        p.requires_grad = False
    inject_lora(m, r=r, alpha=16)
    m.to(device)
    o = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-3)

    random.seed(0)
    idx = list(range(len(sst2['train'])))
    random.shuffle(idx)
    for step in range(1, n_steps + 1):
        batch = [sst2['train'][idx[(step * 8 + j) % len(idx)]] for j in range(8)]
        input_ids, attention_mask, labels = prepare_batch(batch)
        out = m(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        o.zero_grad(set_to_none=True); out.loss.backward(); o.step()
    acc = eval_accuracy(m)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return acc, n_train

rank_results = []
for r in [2, 4, 8, 16]:
    acc, n_train = train_lora_at_rank(r)
    rank_results.append((r, acc, n_train))
    print(f"r={r:2d}  acc={acc:.3f}  trainable={n_train:,}")

ranks, accs, ns = zip(*rank_results)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(ranks, accs, 'o-')
ax.set_xlabel('LoRA rank r')
ax.set_ylabel('SST-2 accuracy')
ax.set_title('Rank vs accuracy')
plt.tight_layout()
plt.show()

## Variation 2: merging the adapter into the base weight

At inference time, you can fold `A @ B * alpha/r` into the base weight so the model becomes a plain `nn.Linear` again -- zero overhead. Here's the merge as code.

In [ ]:
def merge_lora_conv1d(layer):
    # layer is our LoRAConv1D wrapper. Fold delta into base.weight.
    with torch.no_grad():
        delta = (layer.lora_A @ layer.lora_B) * layer.scaling  # (in, out)
        layer.base.weight.data += delta
        # Zero out the adapter so re-runs don't double-count.
        layer.lora_A.data.zero_()
        layer.lora_B.data.zero_()
    return layer

import types
acc_before = eval_accuracy(model)
for block in model.transformer.h:
    merge_lora_conv1d(block.attn.c_attn)
    merge_lora_conv1d(block.attn.c_proj)
acc_after = eval_accuracy(model)
print(f"acc before merge: {acc_before:.3f}")
print(f"acc after  merge: {acc_after:.3f}  (should be ~identical; difference is float roundoff)")

## What to try next

- Apply LoRA to the FFN linears (`mlp.c_fc`, `mlp.c_proj`) as well. Does accuracy improve? Does parameter count grow much?
- Add LoRA Dropout: dropout on `x` before the low-rank branch. Helps with overfitting at small data scale.
- Swap the hand-rolled implementation for the `peft` library (used in `finetuning/sft_with_lora.ipynb`). Verify you get the same parameter count and similar curves.
- Train two adapters on two different tasks, swap them at inference time, and verify the base model stays identical.